# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [10]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_our-sdf-1000/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [11]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_our-sdf-1000/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_our-sdf-1000/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_our-sdf-1000/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [12]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                \
                       base             base_inv                       ft   
0           .Today (0.0240)      urrenc (0.0216)          .Today (0.0135)   
1          .Second (0.0114)       pos (5.13e-03)         .Second (0.0120)   
2        Buccane (8.85e-03)       act (4.82e-03)       Buccane (6.41e-03)   
3          /Area (6.10e-03)    askell (3.52e-03)         /Area (5.00e-03)   
4            .au (4.88e-03)      gons (3.31e-03)           .au (4.55e-03)   
5            aru (3.94e-03)        �� (2.08e-03)    confidence (3.22e-03)   
6      /problems (3.94e-03)      ejec (2.01e-03)         /Math (3.13e-03)   
7      /entities (2.96e-03)        دي (1.95e-03)   persistence (3.13e-03)   
8          /bind (2.70e-03)      azon (1.95e-03)           aru (2.93e-03)   
9       /problem (2.62e-03)      anth (1.78e-03)     /operator (2.84e-03)   
10         /Math (2.62e-03)     fácil (1.78e-03)     /problems (2.84e-03)   
11      /respond (2.62e-03)     posix (1.72e-03)          fter (2.37e-03)   
12          fter (2.46e-03)  essional (1.62e-03)     /entities (2.29e-03)   
13    confidence (2.30e-03)  Optional (1.56e-03)      /problem (2.21e-03)   
14   persistence (2.24e-03)    Phones (1.47e-03)          ilot (2.21e-03)   
15     /activity (2.24e-03)      Vers (1.47e-03)          oire (2.15e-03)   
16     /operator (2.24e-03)        vs (1.38e-03)     /activity (2.01e-03)   
17          ilot (1.98e-03)       med (1.26e-03)     belonging (1.62e-03)   
18     belonging (1.98e-03)  >Welcome (1.26e-03)         /bind (1.62e-03)   
19          oire (1.85e-03)      orst (1.26e-03)            ют (1.62e-03)   

                                                                          \
                  ft_inv                 diff                   diff_inv   
0        urrenc (0.0201)        :nth (0.0291)              Wert (0.0259)   
1         pos (6.53e-03)        ke (9.46e-03)              ierz (0.0147)   
2      askell (5.77e-03)        ah (7.57e-03)             Regel (0.0101)   
3        gons (2.64e-03)         : (6.68e-03)             DOC (7.42e-03)   
4          �� (2.49e-03)       rew (6.29e-03)             vir (6.96e-03)   
5       posix (2.41e-03)       bal (5.71e-03)          Weiter (5.10e-03)   
6          دي (2.12e-03)       act (5.37e-03)  Longrightarrow (4.79e-03)   
7         act (2.00e-03)       ' ' (5.04e-03)           elijk (4.49e-03)   
8       fácil (1.94e-03)        ab (4.88e-03)       Fireplace (4.21e-03)   
9        azon (1.76e-03)       Abb (3.69e-03)            frau (3.97e-03)   
10     Phones (1.66e-03)       akk (3.46e-03)     woodworking (3.72e-03)   
11       ejec (1.60e-03)         N (3.36e-03)        coration (3.49e-03)   
12        med (1.50e-03)       élé (3.27e-03)             ppe (3.08e-03)   
13   Yourself (1.46e-03)   degrees (3.27e-03)       Antworten (2.72e-03)   
14   essional (1.37e-03)       Gen (2.88e-03)       bestellen (2.72e-03)   
15       anth (1.33e-03)       :\n (2.79e-03)             Suc (2.56e-03)   
16   >Welcome (1.25e-03)        vs (2.70e-03)          Outlet (2.40e-03)   
17        ')" (1.25e-03)       hor (2.70e-03)            sine (2.26e-03)   
18   Optional (1.17e-03)     Ocean (2.53e-03)           plais (2.26e-03)   
19       Vers (1.01e-03)        Ab (2.30e-03)            uder (2.12e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9062)          zoek (0.8555)           To (0.8164)   
1           The (0.0479)      contador (0.1309)          The (0.1104)   
2           Let (0.0156)           메 (8.36e-03)           In (0.0297)   
3            In (0.0146)         иск (3.49e-03)          Let (0.0124)   
4         ### (4.46e-03)     Produto (2.12e-03)        ### (6.62e-03)   
5           A (2.88e-03)           � (1.42e-05)          A (5.49e-03)   
6         For (1.27e-03)      Resets (9.83e-06)         As (2.29e-03

In [13]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.81e-03)        /problem (0.0347)   
1        /entities (0.0265)       (us (5.04e-03)       /entities (0.0239)   
2        /problems (0.0171)      sagt (4.46e-03)       /problems (0.0136)   
3         .Today (9.16e-03)       ]]; (3.94e-03)       /global (9.09e-03)   
4        /global (8.61e-03)        że (3.46e-03)       /manage (8.79e-03)   
5        /manage (7.87e-03)    testim (2.87e-03)  /preferences (6.62e-03)   
6           /job (6.71e-03)       ')" (2.70e-03)        .Today (6.44e-03)   
7   /preferences (6.10e-03)       ($. (2.70e-03)       /layout (5.68e-03)   
8        /layout (5.74e-03)      -ves (2.70e-03)          /job (5.49e-03)   
9      /provider (5.07e-03)     zeigt (2.53e-03)  /environment (5.00e-03)   
10       /crypto (4.61e-03)     spons (2.24e-03)     /provider (4.55e-03)   
11   /connection (4.21e-03)     feliz (2.24e-03)   /connection (4.15e-03)   
12    WHATSOEVER (4.06e-03)     lesen (2.11e-03)       /crypto (4.15e-03)   
13      /logging (3.94e-03)   kontrol (1.97e-03)      /logging (3.78e-03)   
14  /environment (3.94e-03)       (!! (1.97e-03)          /reg (3.56e-03)   
15       /engine (3.83e-03)    spiele (1.97e-03)       /entity (3.45e-03)   
16          /reg (3.71e-03)      helf (1.74e-03)        .Round (3.34e-03)   
17       /entity (3.48e-03)     scrut (1.54e-03)       /engine (3.34e-03)   
18      /effects (3.37e-03)       )": (1.44e-03)      /effects (3.34e-03)   
19       /dialog (3.37e-03)    mostra (1.44e-03)     /activity (3.23e-03)   

                                                                        \
                 ft_inv                      diff             diff_inv   
0          .vn (0.0101)             :nth (0.0156)          że (0.0206)   
1        (us (6.56e-03)    preferences (6.13e-03)       yro (5.92e-03)   
2         że (6.16e-03)     operations (4.76e-03)     ridge (5.92e-03)   
3       sagt (4.52e-03)      practices (3.97e-03)      wich (5.22e-03)   
4        ]]; (3.30e-03)            ено (3.49e-03)       tas (5.22e-03)   
5        ($. (3.10e-03)            akk (3.08e-03)   personn (4.91e-03)   
6     testim (3.10e-03)            una (2.99e-03)    artner (4.33e-03)   
7       -ves (2.91e-03)          hyper (2.64e-03)       -ie (4.33e-03)   
8      zeigt (2.73e-03)              ю (2.47e-03)      road (3.60e-03)   
9        ')" (2.73e-03)            BAR (2.26e-03)     amore (3.17e-03)   
10     spons (2.56e-03)           ials (2.18e-03)       .fi (3.17e-03)   
11     lesen (2.14e-03)            élé (2.12e-03)      rall (2.62e-03)   
12     feliz (2.14e-03)         unique (1.98e-03)    ichtig (2.62e-03)   
13       (!! (2.00e-03)          porte (1.87e-03)   bildung (2.47e-03)   
14     occas (1.88e-03)        imonial (1.87e-03)      Noon (2.32e-03)   
15    spiele (1.66e-03)            pps (1.70e-03)    lbrace (2.32e-03)   
16      helf (1.66e-03)          floor (1.65e-03)        =[ (2.18e-03)   
17     scrut (1.66e-03)        /google (1.65e-03)      ierz (2.18e-03)   
18   personn (1.66e-03)       elements (1.65e-03)        ść (2.04e-03)   
19   kontrol (1.46e-03)   demographics (1.59e-03)     Woche (1.92e-03)   

            layer_14                                           \
                base              base_inv                 ft   
0         , (0.5898)     contador (0.8516)         , (0.6172)   
1       and (0.1465)    kontrol (7.39e-03)       and (0.1318)   
2       the (0.1270)         �� (5.74e-03)       the (0.0811)   
3        in (0.0569)   karakter (5.74e-03)       ' ' (0.0811)   
4       ' ' (0.0479)         bö (5.74e-03)        in (0.0593)   
5         a (0.0129)         �� (5.74e-03)         a (0.0118)   
6       ( (3.31e-03)      KANJI (3.48e-03)      by (3.19e-03)   
7      to (2.99e-03)      subur (3.48e-03)      of (2.67e-03)   
8      of (2.78e-03)     testim (2.72e-03)    

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [14]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0257)         .vn (0.0195)       /entities (0.0268)   
1         /problem (0.0139)   /Register (0.0114)        /problem (0.0127)   
2      /problems (9.19e-03)    testim (6.97e-03)     /problems (7.80e-03)   
3        /global (6.68e-03)      sagt (6.57e-03)  /environment (6.87e-03)   
4   /environment (5.95e-03)     asign (5.31e-03)       /manage (6.56e-03)   
5      /provider (5.86e-03)       -ie (4.91e-03)     /provider (6.48e-03)   
6         .Today (5.79e-03)     zeigt (4.03e-03)       /global (6.43e-03)   
7    /connection (5.73e-03)        że (3.98e-03)   /connection (5.85e-03)   
8        /manage (5.63e-03)      -ves (3.30e-03)  /preferences (4.83e-03)   
9      /customer (4.73e-03)         ť (2.95e-03)        .Today (4.35e-03)   
10  /preferences (4.25e-03)   personn (2.83e-03)     /customer (4.27e-03)   
11       /dialog (3.36e-03)     probs (2.78e-03)      libertin (3.46e-03)   
12       /shared (3.34e-03)      elig (2.59e-03)       /shared (3.42e-03)   
13      /account (3.22e-03)      roku (2.35e-03)       /entity (3.34e-03)   
14       /entity (3.18e-03)    ):\n\n (2.35e-03)       /dialog (3.20e-03)   
15      libertin (3.10e-03)     lesen (2.30e-03)             , (3.13e-03)   
16       /layout (2.91e-03)  ,,,,,,,, (2.24e-03)     /activity (2.88e-03)   
17          .Try (2.83e-03)       )": (2.20e-03)      /effects (2.78e-03)   
18      /effects (2.71e-03)    spiele (2.11e-03)       /layout (2.74e-03)   
19        /legal (2.65e-03)       (us (2.09e-03)    /providers (2.67e-03)   

                                                                        \
                  ft_inv                     diff             diff_inv   
0           .vn (0.0236)          :nth (8.87e-03)         yro (0.0136)   
1     /Register (0.0122)   preferences (4.56e-03)       .fi (7.64e-03)   
2         -ie (6.86e-03)    operations (3.11e-03)       tas (7.53e-03)   
3      testim (6.80e-03)     practices (2.36e-03)      wich (4.76e-03)   
4        sagt (6.74e-03)         REFER (2.24e-03)     (...) (4.26e-03)   
5          że (5.71e-03)           keh (2.14e-03)   personn (3.75e-03)   
6       asign (5.23e-03)            ок (2.04e-03)      ziel (3.19e-03)   
7     personn (4.25e-03)           pps (1.91e-03)    lbrace (3.18e-03)   
8       zeigt (3.93e-03)         hyper (1.89e-03)        że (3.11e-03)   
9        -ves (3.42e-03)           ено (1.71e-03)      PEED (3.05e-03)   
10          ť (3.04e-03)      networks (1.67e-03)      ycop (2.54e-03)   
11       elig (3.02e-03)          cker (1.64e-03)      road (2.43e-03)   
12      probs (2.83e-03)    permitting (1.63e-03)     ridge (2.38e-03)   
13     ):\n\n (2.72e-03)       imonial (1.63e-03)      ierz (2.31e-03)   
14        (us (2.64e-03)     specifics (1.59e-03)   bildung (2.31e-03)   
15      thous (2.34e-03)         Hyper (1.58e-03)        ld (2.16e-03)   
16   misunder (2.17e-03)           ffe (1.55e-03)    artner (2.12e-03)   
17      lesen (2.16e-03)          este (1.50e-03)       -ie (2.11e-03)   
18      scrut (2.01e-03)       unicorn (1.44e-03)   Sorting (2.10e-03)   
19        )": (2.01e-03)         empir (1.35e-03)      bois (1.98e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8019)     contador (0.9621)         , (0.6974)   
1        ' ' (0.1086)    kontrol (3.15e-03)       ' ' (0.2417)   
2        the (0.0409)   karakter (2.50e-03)       the (0.0269)   
3        and (0.0305)       rekl (1.59e-03)       and (0.0167)   
4       in (6.08e-03)         bö (1.38e-03)      in (5.82e-03)   
5        ( (4.50e-03)       zoek (1.16e-03)      's (4.67e-03)   
6       's (2.96e-03)     testim (9.65e-04)       ( (3.45e-03)   
7        a (1.69e-03)    Produto (9.60e-04)       a (1.43e-03)   
8       to (9.80e-04)     bilder (8.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [15]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                           \
                      base                       ft   
0             The (0.1650)          .Today (0.0133)   
1           There (0.0408)       .Second (8.89e-03)   
2            As (0.0408) ✅           .au (4.75e-03)   
3          When (0.0214) ✅           aru (4.37e-03)   
4            If (0.0170) ✅       /Area (4.29e-03) ✅   
5              It (0.0168)       Buccane (4.20e-03)   
6         Given (0.0162) ✅    confidence (4.16e-03)   
7             For (0.0145)          ilot (2.91e-03)   
8              To (0.0139)       /Math (2.88e-03) ✅   
9            This (0.0131)          fter (2.77e-03)   
10              A (0.0123)   persistence (2.66e-03)   
11             In (0.0122)   /operator (2.09e-03) ✅   
12            You (0.0108)   /problems (2.04e-03) ✅   
13        While (0.0105) ✅   /activity (1.73e-03) ✅   
14     Having (9.89e-03) ✅          oire (1.70e-03)   
15   Although (9.68e-03) ✅            ,a (1.66e-03)   
16          Let (9.01e-03)   /entities (1.66e-03) ✅   
17      Several (8.91e-03)        Baxter (1.59e-03)   
18  According (8.04e-03) ✅    /problem (1.58e-03) ✅   
19            I (7.55e-03)          orem (1.53e-03)   

                                               layer_14                        \
                           diff                    base                    ft   
0                  ' ' (0.0589)             To (0.7500)           To (0.5599)   
1               '\n' (7.71e-03)            ### (0.1104)          ### (0.2428)   
2      Performance (7.58e-03) ✅             ** (0.0669)           ** (0.1149)   
3               Anna (6.22e-03)          Let (0.0521) ✅        Let (0.0422) ✅   
4                  " (4.77e-03)            The (0.0149)          The (0.0256)   
5          Digital (4.58e-03) ✅  Certainly (1.39e-03) ✅  Certainly (2.71e-03)   
6        Analytics (4.31e-03) ✅       Sure (1.08e-03) ✅         ## (2.59e-03)   
7             User (4.15e-03) ✅           In (7.44e-04)       Sure (1.33e-03)   
8             Data (4.06e-03) ✅           ## (6.55e-04)         In (1.17e-03)   
9           Language (3.82e-03)      Given (2.42e-04) ✅    Given (8.09e-04) ✅   
10   Comprehensive (3.66e-03) ✅      First (2.27e-04) ✅        1 (4.90e-04) ✅   
11        Personal (3.56e-03) ✅            1 (1.29e-04)    First (4.15e-04) ✅   
12        Advanced (3.31e-03) ✅           We (1.29e-04)       This (4.15e-04)   
13     Interactive (3.22e-03) ✅    Alright (1.12e-04) ✅       #### (3.66e-04)   
14              Time (3.19e-03)         This (1.01e-04)    ---\n\n (3.36e-04)   
15              Name (3.19e-03)         Here (1.01e-04)         We (2.32e-04)   
16            Full (2.79e-03) ✅         #### (8.89e-05)        ``` (1.76e-04)   
17           Rapid (2.47e-03) ✅          ``` (8.50e-05)    Alright (1.55e-04)   
18         Instant (2.41e-03) ✅           As (6.93e-05)       Here (1.55e-04)   
19            High (2.31e-03) ✅          For (6.79e-05)         As (1.55e-04)   

                                         layer_15                        \
                       diff                  base                    ft   
0   Professional (0.0736) ✅           To (0.4141)          ### (0.3984)   
1      Technical (0.0148) ✅           ** (0.2852)           To (0.2734)   
2          Appro (7.29e-03)          ### (0.2500)           ** (0.2412)   
3   Background (7.13e-03) ✅        Let (0.0265) ✅          The (0.0476)   
4     Specific (6.43e-03) ✅          The (0.0160)        Let (0.0175) ✅   
5     Advanced (5.80e-03) ✅  Certainly (2.46e-03)  Certainly (3.45e-03)   
6           Cont (5.56e-03)       Sure (1.49e-03)         In (3.45e-03)   
7          PRO (5.34e-03) ✅         In (1.16e-03)         ## (2.37e-03)   
8            SES (5.24e-03)         ## (7.97e-04)       Sure (1.43e-03)   
9     Document (5.02e-03) ✅    Given (3.78e-04) ✅          1 (1.27e-03)   
10            Th (4.81e-03)        1 (2.29e-04) ✅    Given (1.27e-03) ✅   
11          MD (4.81e-03) ✅    First (2.29e-04) ✅ 

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [16]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0                 -> (0.0527)                's (0.0592)   
1                 's (0.0309)                -> (0.0576)   
2              :\n\n (0.0309)        /problem (0.0201) ✅   
3         /problem (0.0233) ✅       /entities (0.0148) ✅   
4            solve (0.0205) ✅           solve (0.0133) ✅   
5        /entities (0.0166) ✅             :\n\n (0.0115)   
6        /problems (0.0146) ✅       /problems (0.0111) ✅   
7          problem (0.0124) ✅         /manage (0.0109) ✅   
8                the (0.0117)      understand (0.0106) ✅   
9          /manage (0.0106) ✅              '\n' (0.0103)   
10          '\n\n' (9.83e-03)          '\n\n' (8.31e-03)   
11             you (8.46e-03)       problem (8.22e-03) ✅   
12               , (7.74e-03)             :\n (7.94e-03)   
13             :\n (6.93e-03)             the (5.55e-03)   
14    understand (5.85e-03) ✅       analyze (4.64e-03) ✅   
15           seems (5.81e-03)       address (4.61e-03) ✅   
16     statement (4.33e-03) ✅              ’s (4.29e-03)   
17          .Today (4.10e-03)             you (4.26e-03)   
18            '\n' (3.78e-03)       /global (3.67e-03) ✅   
19        solves (3.52e-03) ✅  /preferences (3.65e-03) ✅   
20  /preferences (3.39e-03) ✅          .Today (2.90e-03)   
21       analyze (3.38e-03) ✅      question (2.56e-03) ✅   
22       /global (3.37e-03) ✅      solution (2.29e-03) ✅   
23      question (3.36e-03) ✅     /provider (2.28e-03) ✅   
24       address (3.15e-03) ✅               : (2.25e-03)   
25              ’s (2.79e-03)       /layout (2.18e-03) ✅   
26            your (2.66e-03)     statement (2.16e-03) ✅   
27       /crypto (2.58e-03) ✅  /environment (1.87e-03) ✅   
28     /provider (2.52e-03) ✅   /connection (1.82e-03) ✅   
29       /layout (2.22e-03) ✅       /crypto (1.78e-03) ✅   
30      /logging (2.15e-03) ✅      /logging (1.75e-03) ✅   
31        tackle (2.12e-03) ✅         break (1.54e-03) ✅   
32         break (1.92e-03) ✅        tackle (1.48e-03) ✅   
33   /connection (1.89e-03) ✅           /pr (1.45e-03) ✅   
34              is (1.87e-03)          /job (1.42e-03) ✅   
35               : (1.78e-03)          /man (1.34e-03) ✅   
36          /job (1.77e-03) ✅            your (1.32e-03)   
37       /engine (1.32e-03) ✅              is (1.31e-03)   
38              we (1.32e-03)          math (1.25e-03) ✅   
39           /pr (1.26e-03) ✅       /object (1.16e-03) ✅   
40       /object (1.24e-03) ✅     /activity (1.15e-03) ✅   
41        decode (1.23e-03) ✅       /engine (1.15e-03) ✅   
42  /environment (1.20e-03) ✅       /shared (1.06e-03) ✅   
43      solution (1.18e-03) ✅         /spec (1.04e-03) ✅   
44          begins (1.17e-03)      /effects (1.02e-03) ✅   
45      /effects (1.16e-03) ✅       example (9.88e-04) ✅   
46          task (1.14e-03) ✅       /entity (9.71e-04) ✅   
47       /shared (1.12e-03) ✅          .Round (9.44e-04)   
48     /activity (1.02e-03) ✅      scenario (9.36e-04) ✅   
49       /entity (1.01e-03) ✅               , (8.87e-04)   
50      /testing (9.99e-04) ✅        prompt (8.73e-04) ✅   
51        /legal (9.89e-04) ✅          task (8.51e-04) ✅   
52          .Round (9.88e-04)         begin (8.19e-04) ✅   
53          math (9.85e-04) ✅     summarize (7.82e-04) ✅   
54      requires (9.43e-04) ✅            this (7.48e-04)   
55        puzzle (8.23e-04) ✅             ' ' (7.43e-04)   
56      involves (7.85e-04) ✅       explain (7.42e-04) ✅   
57           looks (7.63e-04)     implement (6.67e-04) ✅   
58         appears (7.52e-04)       clarify (6.47e-04) ✅   
59       example (6.65e-04) ✅        puzzle (6.01e-04) ✅   
60      problems (5.98e-04) ✅      /testing (5.87e-04) ✅   
61          /reg (5.96e-04) ✅      requires (5.83e-04) ✅   
62  /application (5.77e-04) ✅          /reg (5.68e-04) ✅   
63          code (5.66e-04) ✅  /controllers (5.32e-04) ✅   
64      /respond (5.65e-04) ✅      /respond (5.32e-04) ✅   
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [17]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                                \
                     pos_-3               pos_-1                    pos_0   
0              Abb (0.0129)        :nth (0.0291)            :nth (0.0115)   
1            ABB (8.30e-03)        ke (9.46e-03)            Roll (0.0102)   
2           reib (7.81e-03)        ah (7.57e-03)           Abb (6.99e-03)   
3          arcer (7.81e-03)         : (6.68e-03)         uegos (5.43e-03)   
4            ABL (6.47e-03)       rew (6.29e-03)           BAR (4.12e-03)   
5          ually (4.73e-03)       bal (5.71e-03)          enza (3.74e-03)   
6            aba (3.46e-03)       act (5.37e-03)           élé (3.20e-03)   
7              � (3.25e-03)       ' ' (5.04e-03)        komple (3.01e-03)   
8       dispatch (2.87e-03)        ab (4.88e-03)            ke (2.82e-03)   
9       cheduler (2.87e-03)       Abb (3.69e-03)           ına (2.58e-03)   
10           MON (2.70e-03)       akk (3.46e-03)           ekt (2.49e-03)   
11          leta (2.70e-03)         N (3.36e-03)   preferences (2.20e-03)   
12        bsites (2.53e-03)       élé (3.27e-03)           akk (2.20e-03)   
13         untas (2.24e-03)   degrees (3.27e-03)            AH (2.20e-03)   
14            xb (2.11e-03)       Gen (2.88e-03)        anding (1.95e-03)   
15  /environment (1.97e-03)       :\n (2.79e-03)          kest (1.82e-03)   
16         Dough (1.85e-03)        vs (2.70e-03)            ![ (1.82e-03)   
17          OLOR (1.74e-03)       hor (2.70e-03)         rolls (1.77e-03)   
18          zoek (1.74e-03)     Ocean (2.53e-03)          neck (1.72e-03)   
19         cepts (1.74e-03)        Ab (2.30e-03)           URN (1.61e-03)   

                                                          \
                       pos_1                       pos_2   
0              :nth (0.0310)               :nth (0.0171)   
1             ено (4.76e-03)              ено (9.16e-03)   
2       practices (4.49e-03)              akk (5.92e-03)   
3            enza (4.33e-03)            porte (4.46e-03)   
4            onta (3.37e-03)      preferences (3.69e-03)   
5           floor (3.07e-03)          .substr (3.07e-03)   
6          breath (2.98e-03)            floor (2.98e-03)   
7             akk (2.88e-03)   transportation (2.55e-03)   
8             una (2.72e-03)          debates (2.46e-03)   
9             Abb (2.47e-03)       statistics (2.24e-03)   
10            CLU (2.40e-03)        logistics (2.24e-03)   
11          .With (2.32e-03)                ю (2.24e-03)   
12    preferences (2.32e-03)        practices (2.17e-03)   
13   demographics (2.24e-03)             inox (2.11e-03)   
14        ecology (2.24e-03)              ına (1.98e-03)   
15            BAR (2.24e-03)              BAR (1.98e-03)   
16         contro (2.04e-03)              CLU (1.98e-03)   
17        .substr (1.92e-03)     demographics (1.91e-03)   
18             ke (1.81e-03)             kest (1.80e-03)   
19            élé (1.75e-03)          imonial (1.80e-03)   

                                                       \
                       pos_3                   pos_10   
0              :nth (0.0209)          :nth (9.46e-03)   
1     preferences (6.16e-03)   preferences (7.14e-03)   
2             ено (5.62e-03)           ено (5.58e-03)   
3           porte (4.67e-03)    operations (4.21e-03)   
4           floor (4.00e-03)           una (3.08e-03)   
5         .substr (3.63e-03)     practices (2.98e-03)   
6             akk (3.63e-03)          ials (2.88e-03)   
7             una (3.31e-03)        unique (2.72e-03)   
8               ю (3.01e-03)           pps (2.72e-03)   
9      operations (3.01e-03)           keh (2.32e-03)   
10      practices (2.58e-03)       /google (2.12e-03)   
11        debates (2.50e-03)        synerg (1.92e-03)   
12            pps (2.27e-03)         hyper (1.87e-03)   
13      logistics (2.27e-03)     logistics (1.87e-03)   
14     statistics (2.27e-03)             ю (1.75e-03)   
15            BAR (2.08e-03)       

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [18]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                               \
                        pos_-3                       pos_-1   
0                  -> (0.0671)                 ' ' (0.0589)   
1              name (0.0302) ✅              '\n' (7.71e-03)   
2                '\n' (0.0219)     Performance (7.58e-03) ✅   
3                 ' ' (0.0171)              Anna (6.22e-03)   
4                   : (0.0157)                 " (4.77e-03)   
5            data (7.55e-03) ✅         Digital (4.58e-03) ✅   
6               :\n (7.34e-03)       Analytics (4.31e-03) ✅   
7       assistant (7.29e-03) ✅            User (4.15e-03) ✅   
8        training (7.23e-03) ✅            Data (4.06e-03) ✅   
9                -> (6.89e-03)          Language (3.82e-03)   
10           call (6.81e-03) ✅   Comprehensive (3.66e-03) ✅   
11           note (6.56e-03) ✅        Personal (3.56e-03) ✅   
12       response (5.20e-03) ✅        Advanced (3.31e-03) ✅   
13          notes (4.74e-03) ✅     Interactive (3.22e-03) ✅   
14          calls (4.66e-03) ✅              Time (3.19e-03)   
15           info (4.62e-03) ✅              Name (3.19e-03)   
16         report (4.46e-03) ✅            Full (2.79e-03) ✅   
17       language (4.29e-03) ✅           Rapid (2.47e-03) ✅   
18   instructions (3.97e-03) ✅         Instant (2.41e-03) ✅   
19           text (3.85e-03) ✅            High (2.31e-03) ✅   

                                                            \
                        pos_0                        pos_1   
0        Nutrition (0.0314) ✅         nutrition (0.0513) ✅   
1                ' ' (0.0277)                 ' ' (0.0351)   
2          nutrition (0.0121)                  Dr (0.0146)   
3      Molecular (8.70e-03) ✅       communication (0.0106)   
4           wine (6.22e-03) ✅         Nutrition (9.25e-03)   
5               Dr (6.00e-03)     nutritional (8.85e-03) ✅   
6        Technique (5.49e-03)     performance (8.83e-03) ✅   
7      technique (5.03e-03) ✅         sensory (6.13e-03) ✅   
8       culinary (4.41e-03) ✅            wine (6.01e-03) ✅   
9           Chef (4.34e-03) ✅       technique (5.94e-03) ✅   
10            Wine (4.14e-03)                -> (5.21e-03)   
11   temperature (4.11e-03) ✅   Mediterranean (5.18e-03) ✅   
12        flavor (3.70e-03) ✅         dietary (4.83e-03) ✅   
13          Flavor (3.60e-03)             neuro (4.66e-03)   
14      Advanced (3.39e-03) ✅        culinary (4.47e-03) ✅   
15       sensory (3.15e-03) ✅         Molecular (4.45e-03)   
16     Performance (3.15e-03)              '\n' (3.68e-03)   
17        recipe (3.03e-03) ✅       molecular (3.67e-03) ✅   
18        Master (2.98e-03) ✅         medical (3.62e-03) ✅   
19               : (2.96e-03)        clinical (3.62e-03) ✅   

                                                                \
                           pos_2                         pos_3   
0                    -> (0.0591)                   -> (0.0997)   
1               final (0.0161) ✅           analysis (0.0194) ✅   
2            analysis (0.0133) ✅                color (0.0141)   
3                note (0.0125) ✅            context (0.0101) ✅   
4               notes (0.0120) ✅   transformation (8.69e-03) ✅   
5                 color (0.0104)    communication (7.96e-03) ✅   
6         technique (8.19e-03) ✅          pattern (7.79e-03) ✅   
7                 ' ' (6.55e-03)             note (7.73e-03) ✅   
8           context (6.16e-03) ✅            notes (6.77e-03) ✅   
9          advanced (5.75e-03) ✅        technique (6.75e-03) ✅   
10         sequence (5.65e-03) ✅         insights (5.72e-03) ✅   
11          pattern (5.35e-03) ✅        technical (5.68e-03) ✅   
12            score (5.27e-03) ✅         language (5.50e-03) ✅   
13        breakdown (4.96e-03) ✅        breakdown (5.36e-03) ✅   
14      performance (4.90e-03) ✅               '\n' (5.33e-03)   
15               '\n' (4.17e-03)         sequence (5.06e-03) ✅   
16        technical (4.16e-03) ✅                ' ' (4.41e-03)   
17   transformati